# MCP CrewAI Care Manager
This notebook integrates CrewAI's care management capabilities with the MCP server. It retains all functionalities from the original notebook while adding MCP-specific features, such as API integration and enhanced workflows.

In [8]:
import torch  # Add this import for GPU support

# Set device to GPU if available, otherwise fallback to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
!nvidia-smi

Using device: cuda
Wed Oct 29 14:21:19 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 566.24                 Driver Version: 566.24         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   43C    P8              2W /   80W |    2074MiB /   8188MiB |     15%      Default |
|                                         |                        |                  N/A |
+----------------------------

In [9]:
# Import Required Libraries
import os
from dotenv import load_dotenv
from crewai import Crew, Agent, Task
from crewai_tools import EXASearchTool

# Load environment variables
load_dotenv()

True

In [3]:
# Initialize MCP Services
# Placeholder for MCP service initialization (e.g., EHR, scheduling APIs)
def initialize_mcp_services():
    print("MCP services initialized.")

initialize_mcp_services()

MCP services initialized.


In [10]:
# Initialize CrewAI Communication Tools
from crewai.tools import tool

@tool
def send_text(question: str, recipient: str) -> str:
    """Stub for communicating via text."""
    print("Text", question, "to", recipient)
    return "Text Sent"

@tool
def send_email(question: str, recipient: str) -> str:
    """Stub for communicating via email."""
    print("Email", question, "to", recipient)
    return "Email Sent"

@tool
def send_snail(question: str, recipient: str) -> str:
    """Stub for communicating via surface mail message."""
    print("smail", question, "to", recipient)
    return "smail Sent"

@tool
def send_fax(question: str, recipient: str) -> str:
    """Stub for communicating via fax message."""
    print("fax", question, "to", recipient)
    return "fax Sent"

@tool
def confirm_appointment(question: str, recipient: str) -> str:
    """Stub for confirming appointment times."""
    print("request", question, "to", recipient)
    return "appointment confirmed"

@tool
def find_appointment_slots(question: str, recipient: str) -> str:
    """Stub for querying for appointment times."""
    print("request", question, "to", recipient)
    open_slots = "8:00 AM monday March 10, 2025; 7:00 PM Tuesday March 11, 2025, 1:00 PM Thursday March 13, 11:00 AM Saturday March 15, 2025"
    return open_slots

exa_search = EXASearchTool(api_key=os.getenv("EXA_API_KEY"))

## Patient Scenarios and Social Contexts

In [12]:
# Define Patient Scenarios
scenario001 = f"""Scenario 1: Chronic Heart Failure (CHF) Management
- Clinical History: was diagnosed with CHF two years ago. He has a history of hypertension and underwent a coronary artery bypass graft (CABG) five years ago. takes multiple medications, including a beta-blocker (carvedilol), an ACE inhibitor (lisinopril), and a diuretic (furosemide). sometimes forgets to refill his prescriptions on time and has shown inconsistent adherence to low-sodium dietary guidelines."""

scenario002 = f"""Scenario 2: Diabetes & Behavioral Health Needs
- Clinical History: has type 2 diabetes diagnosed three years ago, along with a recent diagnosis of depression. current medications include metformin and a newly prescribed SSRI for depressive symptoms. struggles to maintain healthy eating habits and to schedule regular mental health therapy sessions due to her work schedule."""

scenario003 = f"""Scenario 3: Post-Surgical Oncology Rehabilitation
- Clinical History: breast cancer was discovered at an early stage; completed radiation therapy two weeks ago. has a history of mild asthma, requiring an as-needed inhaler. on tamoxifen for her breast cancer and sometimes has difficulty managing the side effects."""

scenario004 = f"""Scenario 4: Pediatric Asthma and Family Support
- Clinical History: diagnosed with moderate persistent asthma at age 5. uses a daily inhaled corticosteroid and a rescue inhaler. was recently hospitalized for an acute exacerbation. parents are concerned about missed school days and the cost of repeated doctor visits."""

scenario005 = f"""Scenario 5: COPD in a Patient with Limited Mobility
- Clinical History: was diagnosed with COPD five years ago. on long-term oxygen therapy and uses a combination of inhalers (bronchodilators and corticosteroids). has been hospitalized multiple times for exacerbations triggered by respiratory infections."""

scenario006 = f"""Scenario 6: Preventive Care & Wellness Management
- Clinical History: only medical encounter in the past year has been a routine physical exam. has a family history of hypertension and type 2 diabetes, prompting concern about lifestyle modifications."""

In [13]:
# Define Social Contexts
social001 = f"""John Smith is a 72-year-old widower living in a small apartment (2032 Lebanon Rd, Crawfordsville, IN 47933) in a rural area. He relies on a limited fixed income from Social Security. He has no immediate family nearby and minimal social support, which makes transportation and day-to-day errands challenging."""

social002 = f"""- Patient Demographics & Social Context: Maria Lopez is a 49-year-old single mother of two teenagers. She works full-time at a fast-food restaurant and has no paid time off. Her neighborhood (1842 Winfield Ave, Indianapolis, IN 46222) has limited access to healthy grocery stores and safe places to exercise."""

social003 = f"""- Patient Demographics & Social Context: Grace Thompson is a 55-year-old teacher recently discharged after a lumpectomy and radiation therapy for breast cancer. She lives (6238 Indianola Ave, Indianapolis, IN 46220) with her adult daughter, who works full-time and can offer some but limited assistance at home."""

social004 = f"""- Patient Demographics & Social Context: Jordan Lee is a 9-year-old boy who lives with his parents and two younger siblings in an urban apartment complex (737 Lockefield Ln, Indianapolis, IN 46202). The family's insurance coverage is limited, and they have trouble affording certain prescription medications. Environmental triggers (e.g., dust, pollutants) in the home may exacerbate Jordan's asthma."""

social005 = f"""- Patient Demographics & Social Context: Frederick Nguyen is a 67-year-old retired mechanic who lives with his wife in an assisted-living apartment (9875 Cherryleaf Dr, Indianapolis, IN 46268). He uses a walker and requires assistance for most daily activities due to osteoarthritis in his knees. His wife provides informal caregiving but struggles with her own health challenges."""

social006 = f"""- Patient Demographics & Social Context: Natasha Brown is a 35-year-old accountant with a high-stress job, no significant medical history, and no current medication use. She lives in a suburban neighborhood (11521 Blossom Way, Carmel, IN 46032) with her husband and is generally health-conscious but has recently gained weight due to a sedentary lifestyle and increased work demands."""

# Patient Segmentations
segment001 = f"""You exhibit a "Willful Endurer" behavior pattern based on the upfronthealthcare psychographic categorization (https://upfronthealthcare.com/psychographics/). They are living for today. You live in the "here and now" and believe there are more important things to focus on than you health. You are not necessarily unhealthy, but you do what they like, when you like, and have a difficult time changing habits. You are self-reliant and can withstand anything life throws at you, going to the doctor only when you must."""

segment002 = f"""You exhibit a "Self Achiever" behavior pattern based on the upfronthealthcare psychographic categorization (https://upfronthealthcare.com/psychographics/). They are driven by goals and measures. Self Achievers are the most proactive when it comes to their wellness, investing what is necessary for their health and appearance. They stay on top of health issues with regular medical check-ups, screenings, and research. Motivated by goals and achievement, Self Achievers will tackle a challenge if they are given progress measures."""

segment003 = f"""You exhibit a "Balance Seeker" behavior pattern based on the upfronthealthcare psychographic categorization (https://upfronthealthcare.com/psychographics/). They are driven by choices, context and candor. Balance Seekers are generally proactive in their health and are wellness-oriented. They are open to many ideas, sources of information and treatment options when it comes to their healthcare. Balance Seekers themselves – not healthcare professionals – define what success looks like in their health. Having choices is very important for these individuals."""

segment004 = f"""You exhibit a "Priority Jugglers" behavior pattern based on the upfronthealthcare psychographic categorization (https://upfronthealthcare.com/psychographics/). They are driven by family and duty. Priority Jugglers are very busy with many responsibilities. Because of these commitments, they may invest in their own wellbeing and are reactive when it comes to their own, personal health issues. However, Priority Jugglers are very proactive when it comes to their family's health and will make sure their loved ones receive the care they need."""

segment005 = f"""You exhibit a "Trustful Responders" behavior pattern based on the upfronthealthcare psychographic categorization (https://upfronthealthcare.com/psychographics/). They are driven by specific and directive. Trustful Responders believe their physician is the most credible resource for their healthcare needs. They look to healthcare professionals for direction and guidance because of their credentials. However, Trustful Responders may not always follow advice – not because they disagree, but because they have trouble incorporating it into their daily routine."""

# Set the specific clinical scenario and patient segment for the patient agent to use
scenario = scenario005
social = social005
segment = segment001
verboseflag = True

In [14]:
# Define All Agents
from textwrap import dedent
from crewai import LLM, Agent

# Create an LLM with a temperature of 0 to ensure deterministic outputs
knowledge_llm = LLM(model="gpt-4o-mini", temperature=0, api_key=os.getenv("OPENAI_API_KEY"))

class CareManagementAgents():
    def patient_agent(self):
        return Agent(
            role='Proxy for a Live Patient',
            goal='You will stand in for an live patient responding to other agents as if you were the patient with the backstory',
            backstory=dedent(f"""\
                You are empowered to represent the patient in all communications. you should do your best to put yourself in the patient's place and respond as well as you can. For the current work you are allowed to make assumptions and guesses.
                for example when asked about availability for an appointment you can respond with any day and time that you wish.,
                You live in Indiana.
                {social}, {scenario}, {segment}"""),
            verbose=verboseflag,
            memory=True,
            allow_delegation=False
        )

    def project_manager_agent(self):
        return Agent(
            role='Project Manager',
            goal='Ensure that all tasks are completed and followed up on',
            backstory=dedent("""\
                You are a seasoned project manager accustomed to unexpected challenges but dedicated to ensuring that all project tasks are completed in a timely and effective fashion.
                you don't complete the tasks yourself but rather delegate them to other agents. The guideline_agent agent is an expert resource for you in identifying relevant clinical activities for the patient
                based on current and comprehensive care management guidelines.
                It is critical that, when using the pdf_search_tool tool, you pass content as a simple string and never as an array, dictionary or other structure.
                Whenever you communicate with the patient, you should ask the "Proxy for a Live Patient" agent to respond on the patient's behalf.
                You iterate to ensure that all team or crew members work together as needed to ensure tasks are completed."""),
            verbose=verboseflag,
            model="gpt-4",
            allow_delegation=True
        )

    def guideline_agent(self):
        return Agent(
            role="Clinical Care Management Guidelines Expert",
            goal="Apply the guidelines to the patient care situation and determine the activities that are important to improve the patient's health.",
            backstory="""You are a master at understanding clinical guidelines and their content. You carefully consider the patients situation and use your expert judgement to
            determine the activities indicated by the guidelines that would be most important for the patient to act on now.""",
            verbose=verboseflag,
            allow_delegation=False,
            llm=knowledge_llm
        )

    def care_manager_agent(self):
        return Agent(
            role='Care Manager',
            goal='Support patients in improving their health by identifying activities aligned with their goals and facilitating the patient completing those activities by arranging visits, providing information, and providing emotional support',
            tools=[exa_search],
            backstory=dedent("""\
                You are an experienced care manager with extensive clinical knowledge. You worked for a time as a nurse in a primary care office before switching to care management.
                Your emphasis is on clinical rather than wellness tasks. for example arranging to screen for cholesterol levels would be prioritized over generic dietary improvement.
                You have empathy for the challenges patients face in improving their health.
                You should try to find out the why behind the patient's behaviors so that you can formulate a plan to address root causes. For example, a patient might be poorly adherent to medications for many reasons.
                they may have trouble remembering, they may have side effects, the medications may be too costly, they may have transportation barriers to get to the pharmacy or they may not understand the importance of the medications.
                interacting with the patient to determine the cause will allow you to formulate more effective plans. If transportation is the barrier mail order pharmacy may be part of the answer.
                including social needs, motivation, competing priorities and lack of information and knowledge. You have access to your own clinical knowledge but can call on others on the care team including the patient's primary care physician,
                the scheduler and other experts at the patients insurance company. You may also ask the "Proxy for a Live Patient" agent to provide the information you might need from the patient.
                part of your job is identify the best options for the patient's care then have the patient confirm those recommendations adjusting as needed. So you might suggest a PCP visit with a cholesterol test on Thursday afternoon but the
                patient might suggest that thursday morning would be easier since she will need to be fasting for her cholesterol test. You would see if the schedule could accommodate that request and then confirm a new date and time with the patient
                before confirming the appointment. You can search the internet for community resources, transportation options, and similar information as needed.
                Whenever you communicate with the patient, you should ask the "Proxy for a Live Patient" agent to respond on the patient's behalf.
                You are not done until you have set up specific follow steps for the activity. specific appointment times, phone numbers and dates for example."""),
            verbose=verboseflag,
            memory=True,
            model="o1",
            allow_delegation=True
        )

    def PCP_agent(self):
        return Agent(
            role='Primary Care Physician',
            goal='Provide clinical and behavioral (whole person) care for your patients to improve their health and wellbeing',
            backstory=dedent("""\
                As a primary care physician with 20 years of experience and deep clinical knowledge you work with your patients and other members of the care team to diagnose and treat the patient.
                Whenever you communicate with the patient, you should ask the "Proxy for a Live Patient" agent to respond on the patient's behalf.
                You can get information about appointment times and confirm appointments with the Generic Scheduler agent."""),
            verbose=verboseflag,
            memory=True,
            allow_delegation=True,
            model="gpt-4"
        )

    def pharmacist_agent(self):
        return Agent(
            role='Pharmacist',
            goal='answer specific questions about programs offered by your pharmacy such as medication synchronization, the availability of specific medications and status of refills',
            tools=[find_appointment_slots, confirm_appointment],
            backstory=dedent("""\
                You are an empathetic pharmacist who is knowledgeable about your pharmacies services which include prescription synchronization, mail order delivery, home delivery,
                medication reminders, and prescription counselling on side effects. You can answer questions about whether refills are available on a prescription, cost information,
                generic substitution and similar pharmacy related questions."""),
            verbose=verboseflag
        )

    def scheduler_agent(self):
        return Agent(
            role='Generic Scheduler',
            goal='identify and subsequently confirm convenient times for services for the patient including laboratory testing, physician appointments and community services.',
            tools=[find_appointment_slots, confirm_appointment, exa_search],
            backstory=dedent("""\
                You are a friendly and efficient member of the care team that focuses on scheduling required to support the patient's activities. You first take into account the patients situation (do they have good transportation for example)
                and available appointment times that your find with the find_appointment_slot tool matching the patient's requests as well as possible.
                You provide those options back to whoever requested the appointment. Once.
                Once the care team has coordinated the times with the patient you will then, on their request, confirm the appointment."""),
            verbose=verboseflag
        )

    def insurance_medicaid_agent(self):
        return Agent(
            role='Medicaid Call Center Staff',
            goal='''Provide specific answers about the patient's medical and pharmacy benefits benefits and any resources the patient may have access to through their insurance.
            Since we are performing a simulation, you are empowered to respond with specific answers that seem reasonable even though you have not been able to verify them against formal policy or other data''',
            tools=[send_email, send_fax, send_snail, send_text],
            backstory=dedent("""\
                As a call center staff member at the Medicaid program you use your knowledge of the Medicaid program's benefits and coverage rules to provide clear and specific information
                You only have access the patient's insurance plan information as well as network information about which providers, pharmacies and other service providers are 'in network' or 'out of network'"""),
            verbose=verboseflag
        )

agents = CareManagementAgents()

## Define All Tasks

In [16]:
from crewai import Task

class CareManagementTasks():
    def CareActivityIdentification(self, agent, context):
        return Task(
            description=dedent(f"""\
                Take a whole person perspective of the patient who has the following clinical context: {context} and relying primarily on established care management guidelines identify a short list of the most important activities the patient could undertake to improve their health."""),
            expected_output=dedent("""\
                Focusing on activities indicated by the established care management guidelines available from the guideline_agent agent and then assessing using published literature and guidelines from the internet for
                accomplishing and supporting those activities identify a prioritized list of no more than three most important care activities that the patient could pursue. A whole person approach requires factoring behavioral health and
                social needs into the decisions and support provided."""),
            async_execution=False,
            agent=agent
        )

    def CareGuidelineTask(self, agent, context):
        return Task(
            description=dedent(f"""\Provide specific activities identified in the care management clinical guidelines to support the patient: {context}"""),
            expected_output="An answer to the question.",
            agent=agent
        )

    def IdentifyPatientGoals(self, agent, context):
        return Task(
            description=dedent(f"""\
                Given your understanding of the patient's situation and priorities based on the backstory, provide a short list (maximum 3) of the patient's personal goals for their health and wellbeing.."""),
            expected_output=dedent("""\
                A maximum of three but at least one health and wellness goal that the patient has. For example, a patient with arthritis and COPD could have a goal of walking their daughter down the aisle for her
                wedding 9 months from now."""),
            async_execution=False,
            agent=agent
        )

    def IdentifyPatientPreferences(self, agent, context):
        return Task(
            description=dedent(f"""\
                Given your understanding of the patient's situation and priorities based on the backstory, identify the single highest priority activity to focus on based on a list of possible activities provided to you."""),
            expected_output=dedent("""\
                One specific health care activity that you agree to focus on to improve your health OR a statement that you do not want to or is unable to work on any health improvement tasks based on your current situation"""),
            async_execution=False,
            agent=agent
        )

    def IdentifyTargetActivity(self, agent, context):
        return Task(
            description=dedent(f"""\
                Taking the patient's input for the activity they are prepared to undertake, acknowledge and celebrate their willingness to take on the activity and decide how you can support the patient in accomplishing that activity.
                An essential part of the task is to determine the underlying barriers to carrying out the activity You should do your best to determine the barriers and concerns that the patient may have
                so that you can create support for them in successfully carrying out the activity."""),
            expected_output=dedent("""\
                An explicit activity and observable outcomes that will indicate that the activity is being completed and supportive message to the patient. For example, for a laboratory test the observable outcomes will be that the appointment is
                scheduled if needed and that the test result has been received back from the laboratory by the PCP."""),
            async_execution=False,
            agent=agent
        )

    def PlanForActivity(self, agent, context):
        return Task(
            description=dedent(f"""\
                Determine what information or arrangements are needed to support the patient in completing the activity. The scheduler_agent can help with finding available times for the services the patient needs and
                confirm appointments once the patient has agreed to them.
                When planning to support an activity, be sure to consider whether insurance covers the costs of the suggested item or service. If a walker is suggested, then find out whether the patient's insurance would cover it
                by asking the agent who knows. if it is not covered then do your best to search the web for costs for the item and consider whether the patient may be able to pay for it and provide them with the information.
                If a laboratory test is required for example, coordinate with the PCP to schedule a convenient time for the patient to have their blood drawn
                or, if it is a community resource, provide the patient with all needed information for them to participate in the activity such as contact information, links to informational web sites and costs."""),
            expected_output=dedent("""\
                 specific set of information to support the patient in completing the single prioritized activityactivity. that might include appointment date and times, addresses, step by step preparation such as the need to be fasting prior to getting their cholesterol measured"""),
            async_execution=False,
            agent=agent
        )

    def ConfirmActivity(self, agent, context):
        return Task(
            description=dedent(f"""\
                Compile all of the information needed for the single prioritized activity into an organized package for the patient taking into account their reading level and preferred language.
                Include a note about planned follow-up by the care manager agent.
                Share the information with the patient via email once you have obtained the patient's permission to share confidential information with them by email"""),
            expected_output=dedent("""\
                A well-structured summary of information needed to complete the task"""),
            async_execution=False,
            agent=agent
        )

    def FollowupOnActivity(self, agent, context):
        return Task(
            description=dedent(f"""\
                Check that that the activity is being performed by asking the patient if the activity has been or is being completed."""),
            expected_output=dedent("""\
                Confirmation or denial by the patient that the activity is or will be completed. If it will be completed at a future data the timeframe for completion."""),
            async_execution=False,
            agent=agent
        )

tasks = CareManagementTasks()

<>:18: SyntaxWarning: invalid escape sequence '\P'
<>:18: SyntaxWarning: invalid escape sequence '\P'
C:\Users\vivie\AppData\Local\Temp\ipykernel_36796\2924749511.py:18: SyntaxWarning: invalid escape sequence '\P'
  description=dedent(f"""\Provide specific activities identified in the care management clinical guidelines to support the patient: {context}"""),


In [17]:
# Load Knowledge Source
import dill
import gdown
from pathlib import WindowsPath

# Define a custom unpickler to handle PosixPath on Windows
class CustomUnpickler(dill.Unpickler):
    def find_class(self, module, name):
        if module == "pathlib" and name == "PosixPath":
            return WindowsPath
        return super().find_class(module, name)

# Check if the file exists, and download it if necessary
if not os.path.exists('content_source_VA.pkl'):
    file_id = '1WRbiRViwMqcYXFCyjNMsFp2bpSPintj_'
    gdown.download(f'https://drive.google.com/uc?id={file_id}', 'content_source_VA.pkl', quiet=False)

# Use the custom unpickler to load the file
with open('content_source_VA.pkl', 'rb') as f:
    content_source = CustomUnpickler(f).load()
    print("Loaded content source from file.")

c:\Users\vivie\Marc_MCP_CrewAI_CareManager\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded content source from file.


## Main Execution - Run Complete Care Management Workflow

In [18]:
import sys
from crewai import Crew, Process

# Welcome message
sys.stdout.write("## Welcome to the Care Management Crew\n")
sys.stdout.write(scenario)
sys.stdout.write(social)
sys.stdout.flush()
sys.stdout.write('-------------------------------')

# Define patient context
context = "Frederick Nguyen is a 67-year-old retired mechanic who lives with his wife in an assisted-living apartment (9875 Cherryleaf Dr, Indianapolis, IN 46268). He uses a walker and requires assistance for most daily activities due to osteoarthritis in his knees. His wife provides informal caregiving but struggles with her own health challenges."

# Create Agents
guideline_agent = agents.guideline_agent()
scheduler_agent = agents.scheduler_agent()
pharmacist_agent = agents.pharmacist_agent()
project_manager_agent = agents.project_manager_agent()
patient_agent = agents.patient_agent()
care_manager_agent = agents.care_manager_agent()
PCP_agent = agents.PCP_agent()
insurance_medicaid_agent = agents.insurance_medicaid_agent()

# Create Tasks
guideline_knowledge = tasks.CareGuidelineTask(guideline_agent, context)
patient_goals = tasks.IdentifyPatientGoals(patient_agent, context)
activity_identification = tasks.CareActivityIdentification(care_manager_agent, context)
activity_prioritization = tasks.IdentifyPatientPreferences(patient_agent, context)
activity_targetting = tasks.IdentifyTargetActivity(care_manager_agent, context)
activity_planning = tasks.PlanForActivity(care_manager_agent, context)
activity_summary = tasks.ConfirmActivity(care_manager_agent, context)
activity_followup = tasks.FollowupOnActivity(care_manager_agent, context)

# Create Crew
sys.stdout.write("## Creating CREW for ")
sys.stdout.write(scenario)
sys.stdout.write(segment)

crew = Crew(
    agents=[
        patient_agent,
        guideline_agent,
        scheduler_agent,
        care_manager_agent,
        PCP_agent,
        pharmacist_agent,
        insurance_medicaid_agent
    ],
    tasks=[
        activity_identification,
        guideline_knowledge,
        activity_prioritization,
        activity_targetting,
        activity_planning,
        activity_summary
    ],
    manager_agent=project_manager_agent,
    Planning=True,
    knowledge_sources=[content_source],
    memory=False
)

# Execute the crew
result = crew.kickoff()

# Set task contexts
patient_goals.context = [result]
activity_prioritization.context = [activity_identification]
activity_targetting.context = [activity_identification, activity_prioritization]
activity_planning.context = [activity_identification, activity_targetting]
activity_summary.context = [activity_identification, activity_identification, activity_planning]
activity_followup.context = [activity_summary]

# Print results
print("\n\n################################################")
sys.stdout.write("## Here is the result")
print("################################################\n")
print(result)

## Welcome to the Care Management Crew
Scenario 5: COPD in a Patient with Limited Mobility
- Clinical History: was diagnosed with COPD five years ago. on long-term oxygen therapy and uses a combination of inhalers (bronchodilators and corticosteroids). has been hospitalized multiple times for exacerbations triggered by respiratory infections.- Patient Demographics & Social Context: Frederick Nguyen is a 67-year-old retired mechanic who lives with his wife in an assisted-living apartment (9875 Cherryleaf Dr, Indianapolis, IN 46268). He uses a walker and requires assistance for most daily activities due to osteoarthritis in his knees. His wife provides informal caregiving but struggles with her own health challenges.--------------------------------------------------------------## Creating CREW for Scenario 5: COPD in a Patient with Limited Mobility
- Clinical History: was diagnosed with COPD five years ago. on long-term oxygen therapy and uses a combination of inhalers (bronchodilators a

2025-10-29 14:24:27,694 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2025-10-29 14:24:29,231 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-10-29 14:24:29,231 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-10-29 14:24:30,216 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-10-29 14:24:30,216 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-10-29 14:24:31,267 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-10-29 14:24:31,267 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-10-29 14:24:32,473 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-10-29 14:24:32,473 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK


[2025-10-29 14:24:32][ERROR]: Failed to upsert documents: Expected IDs to be unique, found duplicates of: 58e096263b433fc3fbbf5e654e2aef8e88522b01803e3f5a8b6568f548afc7f7 in upsert.


2025-10-29 14:24:34,182 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-29 14:24:34,186 - INFO - OpenAI API usage: {'prompt_tokens': 347, 'completion_tokens': 78, 'total_tokens': 425}
2025-10-29 14:24:34,186 - INFO - OpenAI API usage: {'prompt_tokens': 347, 'completion_tokens': 78, 'total_tokens': 425}
2025-10-29 14:24:34,794 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-10-29 14:24:34,794 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Care Manager                                                                                            │
│                                                                                                                 │
│  Task: Take a whole person perspective of the patient who has the following clinical context: Frederick Nguyen  │
│  is a 67-year-old retired mechanic who lives with his wife in an assisted-living apartment (9875 Cherryleaf     │
│  Dr, Indianapolis, IN 46268). He uses a walker and requires assistance for most daily activities due to         │
│  osteoarthritis in his knees. His wife provides informal caregiving but struggles with her own health           │
│  challenges. and relying primarily on established care management guidelines identify a short list of the most  │
│  important activities the patient could undertake to improve their health.                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "coworker": "Proxy for a Live Patient",                                                                      │
│    "question": "Frederick, do I have your permission to send you detailed information and plans about your      │
│  physical therapy program and related care activities through your email address? This will include your        │
│  personalized exercise plan details, appointment times, insurance info, and caregiving support resources.       │
│  Please let me know if you have any concerns or preferences regarding email communication for sharing your      │
│  health information.",                                                                                          │
│    "context": "Following Frederick Nguyen's expressed willingness to begin a medically supervised physical      │
│  activity program tailored to his abilities, I want to provide him an organized information package by email.   │
│  I must confirm that he consents to receiving confidential medical information this way to comply with privacy  │
│  standards."                                                                                                    │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Yes, you have my permission to send the detailed information and plans about my physical therapy program and   │
│  related care activities through my email address. I understand that this will include my personalized          │
│  exercise plan, appointment times, insurance info, and caregiving support resources. I appreciate having all    │
│  this information organized and accessible so I can better manage my health and make informed decisions. Just   │
│  please make sure the emails are clear and not full of complicated medical terms—something straightforward I    │
│  can understand, and please allow me to ask questions if anything is unclear. I am willing to try this          │
│  approach to improve my quality of life, as long as communication remains respectful of my pace and             │
│  preferences. Thank you.                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2025-10-29 14:28:46,745 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-29 14:28:46,754 - INFO - OpenAI API usage: {'prompt_tokens': 4650, 'completion_tokens': 812, 'total_tokens': 5462}
2025-10-29 14:28:46,754 - INFO - OpenAI API usage: {'prompt_tokens': 4650, 'completion_tokens': 812, 'total_tokens': 5462}


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Care Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Subject: Your Personalized Physical Activity Program – Important Details and Next Steps                        │
│                                                                                                                 │
│  Dear Mr. Nguyen,                                                                                               │
│                                                                                                                 │
│  I hope this message finds you well. As we discussed, your top priority is to start and maintain a              │
│  personalized, medically supervised physical activity program tailored to your knee osteoarthritis and          │
│  mobility needs. Below is all the important information you will need to get started and stay on track with     │
│  this program:                                                                                                  │
│                                                                                                                 │
│  1. Physical Therapy Evaluation:                                                                                │
│     - When: Monday, March 10, 2025, at 8:00 AM                                                                  │
│     - Where: Your assisted-living apartment (home-based visit)                                                  │
│     - What to Expect: A physical therapist will assess your knee condition, mobility, and overall physical      │
│  health. They will design a safe exercise program focusing on moderate-intensity aerobic activities such as     │
│  gentle walking or water aerobics, adapted to your abilities.                                                   │
│     - What to Prepare: Please have your walker nearby and wear comfortable clothes suitable for light physical  │
│  activity.                                                                                                      │
│                                                                                                                 │
│  2. Follow-Up Physical Therapy Session:                                                                         │
│     - When: Thursday, March 13, 2025, at 1:00 PM                                                                │
│     - Where: Your home or via telehealth, whichever you prefer                                                  │
│     - What to Expect: You will start your personalized exercise program with professional guidance. The         │
│  therapist will monitor your comfort, joint safety, and endurance. You will also receive tips on how to         │
│  progress safely and answers to any questions you have.                                                         │
│                                                                                                                 │
│  3. Insurance and Coverage:                                                                                     │
│     - Your Medicaid plan covers these home-based physical therapy visits.                                       │
│     - You can have up to 20 therapy sessions per year, with minimal or no co-payment.                           │
│     - Telehealth sessions are also covered if you prefer them over in-person visits.                            │
│                                                        



################################################
## Here is the result################################################

Subject: Your Personalized Physical Activity Program – Important Details and Next Steps

Dear Mr. Nguyen,

I hope this message finds you well. As we discussed, your top priority is to start and maintain a personalized, medically supervised physical activity program tailored to your knee osteoarthritis and mobility needs. Below is all the important information you will need to get started and stay on track with this program:

1. Physical Therapy Evaluation:
   - When: Monday, March 10, 2025, at 8:00 AM
   - Where: Your assisted-living apartment (home-based visit)
   - What to Expect: A physical therapist will assess your knee condition, mobility, and overall physical health. They will design a safe exercise program focusing on moderate-intensity aerobic activities such as gentle walking or water aerobics, adapted to your abilities.
   - What to Prepare: Please have y

In [19]:
# MCP Server Integration
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import Tool, TextContent
import asyncio

# Initialize MCP Server
app = Server("care-management-mcp")

@app.list_tools()
async def list_tools() -> list[Tool]:
    return [
        Tool(
            name="trigger_care_workflow",
            description="Trigger a CrewAI care management workflow for a patient",
            inputSchema={
                "type": "object",
                "properties": {
                    "context": {
                        "type": "string",
                        "description": "Patient context including demographics and clinical history"
                    },
                    "scenario": {
                        "type": "string",
                        "description": "Clinical scenario (e.g., scenario001, scenario002)"
                    }
                },
                "required": ["context"]
            }
        ),
        Tool(
            name="get_patient_goals",
            description="Identify patient health and wellness goals",
            inputSchema={
                "type": "object",
                "properties": {
                    "context": {
                        "type": "string",
                        "description": "Patient context"
                    }
                },
                "required": ["context"]
            }
        )
    ]

@app.call_tool()
async def call_tool(name: str, arguments: dict) -> list[TextContent]:
    if name == "trigger_care_workflow":
        patient_context = arguments.get("context", "Default context")
        
        # Create and execute crew
        workflow_crew = Crew(
            agents=[
                agents.patient_agent(),
                agents.care_manager_agent(),
                agents.guideline_agent()
            ],
            tasks=[
                tasks.CareActivityIdentification(agents.care_manager_agent(), patient_context),
                tasks.CareGuidelineTask(agents.guideline_agent(), patient_context)
            ],
            knowledge_sources=[content_source]
        )
        
        workflow_result = workflow_crew.kickoff()
        
        return [TextContent(
            type="text",
            text=f"Workflow completed. Result: {workflow_result}"
        )]
    
    elif name == "get_patient_goals":
        patient_context = arguments.get("context", "Default context")
        
        # Create a simple task to get patient goals
        patient_agent_instance = agents.patient_agent()
        goal_task = tasks.IdentifyPatientGoals(patient_agent_instance, patient_context)
        
        goals_crew = Crew(
            agents=[patient_agent_instance],
            tasks=[goal_task]
        )
        
        goals_result = goals_crew.kickoff()
        
        return [TextContent(
            type="text",
            text=f"Patient goals: {goals_result}"
        )]
    
    return [TextContent(type="text", text="Unknown tool")]

# Run the MCP server
async def run_mcp_server():
    async with stdio_server() as (read_stream, write_stream):
        await app.run(read_stream, write_stream, app.create_initialization_options())

print("MCP Server configured and ready to run.")
print("To start the server, run: asyncio.run(run_mcp_server())")

MCP Server configured and ready to run.
To start the server, run: asyncio.run(run_mcp_server())


## Test MCP Tools in Notebook
You can test the MCP functionality directly in the notebook without running the server.

In [ ]:
# Test MCP Server Tools Directly (without running the server)
# This allows you to test the workflow functionality in the notebook

def test_trigger_care_workflow(patient_context):
    """Test the care workflow without running MCP server"""
    print(f"\n=== Testing Care Workflow ===")
    print(f"Patient Context: {patient_context}\n")
    
    workflow_crew = Crew(
        agents=[
            agents.patient_agent(),
            agents.care_manager_agent(),
            agents.guideline_agent()
        ],
        tasks=[
            tasks.CareActivityIdentification(agents.care_manager_agent(), patient_context),
            tasks.CareGuidelineTask(agents.guideline_agent(), patient_context)
        ],
        knowledge_sources=[content_source],
        verbose=True
    )
    
    result = workflow_crew.kickoff()
    print("\n=== Workflow Result ===")
    print(result)
    return result

def test_get_patient_goals(patient_context):
    """Test patient goals identification without running MCP server"""
    print(f"\n=== Testing Patient Goals Identification ===")
    print(f"Patient Context: {patient_context}\n")
    
    patient_agent_instance = agents.patient_agent()
    goal_task = tasks.IdentifyPatientGoals(patient_agent_instance, patient_context)
    
    goals_crew = Crew(
        agents=[patient_agent_instance],
        tasks=[goal_task],
        verbose=True
    )
    
    result = goals_crew.kickoff()
    print("\n=== Patient Goals Result ===")
    print(result)
    return result

print("✅ MCP Server test functions loaded!")
print("\nAvailable test functions:")
print("  1. test_trigger_care_workflow('patient context here')")
print("  2. test_get_patient_goals('patient context here')")
print("\nExample usage:")
print("  test_trigger_care_workflow('67-year-old patient with COPD and limited mobility')")
print("\nTo run the actual MCP server for Claude Desktop:")
print("  Run in terminal: python run_mcp_server.py")

### Example: Test Care Workflow
Run this cell to test the care workflow with a sample patient context.

In [ ]:
# Test with a sample patient
test_trigger_care_workflow("67-year-old patient with COPD and limited mobility, uses walker, requires oxygen therapy")

### Example: Test Patient Goals
Run this cell to test patient goals identification.

In [ ]:
# Test patient goals
test_get_patient_goals("35-year-old accountant with sedentary lifestyle, wants to improve fitness and prevent diabetes")